In [9]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from joblib import Parallel, delayed
from matplotlib.ticker import PercentFormatter
import re

In [5]:
nacc_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/comparison_inc')

In [ ]:
def option_string_to_dict(options):
    pattern = r"([A-Z])\. ([^\n]+)"
    matches = re.findall(pattern, options)
    return {key: value for key, value in matches}


def load_answers(dir_path, dataset_name):
    # load all parquet files from the directory, stack them into a pandas datafame
    # this only reads the participant ID, ground trush answer and the prediction. 
    # Reading only those columns is significantly faster (about 100x) than loading the whole dataframe.
    # Loading everything is very slow because there are extremely long strings (model outputs) in some columns

    fpaths = list(dir_path.rglob('*.parquet'))

    dfs = []

    cols_to_read = ['ID','ground_truth_text','ground_truth','prediction','options']

    for fpath in tqdm(fpaths):

        model = fpath.parent.name.split('-',3)[-1] 
        
        if 'test_' in fpath.parent.parent.name:

            benchmark = fpath.parent.parent.name.split('_',1)[-1].upper()

            df = pd.read_parquet(fpath,columns=cols_to_read)
        
            df = df.assign(model=model, benchmark=benchmark)

            df['correct'] = (df['ground_truth'] == df['prediction']).astype(int)
            
            df['prediction_text'] = df.apply(lambda row: option_string_to_dict(row['options']).get(row['prediction'],'invalid'),axis=1)

            dfs.append(df)

    df = pd.concat(dfs)
    df['dataset'] = dataset_name

    # make these columns Categorical
    group_cols = ["dataset", "benchmark", "model", "ground_truth_text", 'prediction_text']
    for col in group_cols:
        df[col] = pd.Categorical(df[col])

    return df


In [36]:
nacc_res = load_answers(nacc_path,dataset_name='NACC')

  0%|          | 0/80 [00:00<?, ?it/s]

100%|██████████| 80/80 [00:05<00:00, 15.44it/s]


In [37]:
def _process_group(id, group, n_boot, metric_names, seed):
    """
    Helper function for parallel processing.
    Computes bootstrap CIs for a single group.
    """
    dataset, benchmark, model = id
    n_samples = len(group)
    
    if n_samples == 0:
        return []

    unique_labels = group['ground_truth_text'].unique()
    
    # --- Reproducible RNG per worker ---
    # Create a new, independent generator seeded for this group
    rng = np.random.default_rng(seed)
    
    # Pre-generate all bootstrap indices at once
    bootstrap_indices = rng.integers(0, n_samples, size=(n_boot, n_samples))
    
    # Pre-allocate dicts to store results (using lists is fine)
    boot_results = {name: [] for name in ['precision', 'recall', 'f1']}
    
    # Convert to NumPy arrays ONCE per group
    y_true = group["ground_truth_text"].to_numpy()
    y_pred = group["prediction_text"].to_numpy()
    
    for indices in bootstrap_indices:
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]

        # we get the metrics per class, then average them after 
        p, r, f, s = precision_recall_fscore_support(
            y_true_boot,
            y_pred_boot,
            average=None, # Get per-class metrics
            labels=unique_labels,
            zero_division=0,
        )
        
        # Manually compute macro-average and total support
        # boot_results["precision"].append(hmean(p))
        # boot_results["recall"].append(hmean(r))
        # boot_results["f1"].append(hmean(f))

        # # Manually compute macro-average and total support
        boot_results["precision"].append(np.mean(p))
        boot_results["recall"].append(np.mean(r))
        boot_results["f1"].append(np.mean(f))


    # --- Calculate quantiles ---
    res_list = []
    for metric_name in metric_names:
        # Convert list to array here
        metric_values = np.array(boot_results[metric_name])
        
        # Use np.partition for O(n) percentile calculation
        low_idx = int(0.025 * n_boot)
        med_idx = int(0.5 * n_boot)
        high_idx = int(0.975 * n_boot)
        
        partitioned = np.partition(metric_values, [low_idx, med_idx, high_idx])
        
        res_list.append(
            {
                "dataset": dataset,
                "benchmark": benchmark,
                "model": model,
                "metric": metric_name,
                "median": partitioned[med_idx],
                "low": partitioned[low_idx],
                "high": partitioned[high_idx],
            }
        )
        
    return res_list


def optimized_bootstrap_parallel(df, metric_names, n_boot, seed=None, n_jobs=-1):
    """
    Compute bootstrap confidence intervals in parallel using joblib.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe
    metric_names : list of str
        List of metric names: ['precision', 'recall', 'f1']
    n_boot : int
        Number of bootstrap samples
    seed : int, optional
        Random seed for reproducible results
    n_jobs : int, optional
        Number of CPU cores to use. -1 means all cores.
    
    Returns
    -------
    pd.DataFrame
        Results with confidence intervals
    """
    
    # --- Setup ---
    # Create a main RNG to generate seeds for each worker
    main_rng = np.random.default_rng(seed)
    
    # Optimization: Convert to categorical for faster groupby
    # We copy to avoid modifying the user's original DataFrame
    df_copy = df.copy()
    group_cols = ["dataset", "benchmark", "model"]
            
    # Get all groups. Must cast to list() to get a concrete list for Parallel
    groups = list(df_copy.groupby(group_cols, observed=True))
    
    # Generate a unique, independent seed for each group
    n_groups = len(groups)
    group_seeds = main_rng.integers(0, 2**32, size=n_groups)
    
    # --- Parallel Execution ---
    results_list_of_lists = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(_process_group)(
            id, 
            group, 
            n_boot, 
            metric_names, 
            group_seeds[i] # Pass a unique seed to each job
        )
        for i, (id, group) in enumerate(groups)
    )
    
    # --- Collect Results ---
    # Flatten the list of lists returned by Parallel
    final_results = [item for sublist in results_list_of_lists for item in sublist]
    
    return pd.DataFrame(final_results)

In [38]:
nacc_metrics = optimized_bootstrap_parallel(
    df=nacc_res,
    metric_names=['precision', 'recall', 'f1'],
    n_boot=10,
    seed=42,
    n_jobs=32
)

In [39]:
sahana_path = Path(
    "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/llm_answer_extractor/outputs_access"
)


def load_sahana_data(path):
    csv_files = list(path.glob("*.csv"))

    dfs = []

    for file in csv_files:
        df = pd.read_csv(file)

        df["benchmark"] = file.stem.upper()
        df = df.rename(columns={"score":'pass@1'})

        dfs.append(df)

    return pd.concat(dfs).reset_index(drop=True)


model_name_map = {
    "qwen25_3B": "Qwen2.5-3B-Instruct",
    "qwen25_3B_drgrpo_gp16_nacc_inc": "NACC_Inc-1000",
    "qwen25_3B_drgrpo_gp16_nacc_inc_sce_tanh": "NACC_Inc-sce-tanh-1000",
    "qwen25_3B_drgrpo_gp16_nacc_inc_oversample_sce_tanh": "NACC-inc-os-sce",
    "qwen25_3B_drgrpo_gp16_nacc_inc_oversample": "NACC-inc-os",
}

bench_name_map = {
    'NEUROPATH': 'NP',
    'COGSTAT': 'COG',
    'AMYLPET': 'PET'
}

sahana_res = load_sahana_data(sahana_path)

sahana_res = sahana_res[sahana_res['model'].isin(model_name_map.keys())]

sahana_res = sahana_res.replace(model_name_map).replace(bench_name_map)

In [40]:
df = nacc_metrics.sort_values(
    [
        "dataset",
        "benchmark",
        "model",
        "metric",
    ]
)

df = df[~df["model"].isin(["Qwen3-4B","HuatuoGPT-o1-8B"])]

matteo_res = df[df['metric'] == 'recall']

In [41]:
df = pd.merge(sahana_res,matteo_res[['benchmark','model','median','low','high']], on=['benchmark','model'],how='inner')

df = df[['benchmark','model','pass@1','median','low','high']].sort_values(['benchmark','model']).round(3)

In [42]:
df.head()

,benchmark,model,pass@1,median,low,high
22,COG,NACC-inc-os,0.755,0.758,0.754,0.762
23,COG,NACC-inc-os-sce,0.765,0.769,0.761,0.771
21,COG,NACC_Inc-1000,0.753,0.757,0.751,0.762
24,COG,NACC_Inc-sce-tanh-1000,0.766,0.767,0.760,0.774
20,COG,Qwen2.5-3B-Instruct,0.679,0.668,0.660,0.672


In [43]:
import plotly.express as px
import plotly.graph_objects as go

In [46]:
fig = px.scatter(
    df,
    x='pass@1',
    y='median',
    color='benchmark',
    symbol='model',
    width=800,
    height=600
)

# Determine the limits for the bisector line
# This ensures the bisector spans the visible range of your data
min_val = min(0.2, 0.2)
max_val = max(0.8,0.8)

# Add the bisector line (y=x) as a new scatter trace
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='Bisector (y=x)',
    line=dict(color='black', dash='dash') # Customize line appearance
))

# Show the plot
fig.show()